In [2]:
!python scripts/generate_dataset.py

In [1]:
import json
train_data = []
with open('data/train.jsonl', 'r') as f:
    for line in f.readlines():
        if line.strip():
            train_data.append(json.loads(line))

print(train_data[0])

{'expression': '25 - 14', 'answer': 11, 'type': 'standard'}


In [2]:
import os
os.environ.keys()
os.environ["GOOGLE_CLOUD_PROJECT"] = os.environ["ANTHROPIC_VERTEX_PROJECT_ID"]
os.environ["GOOGLE_CLOUD_REGION"] = os.environ["CLOUD_ML_REGION"]

In [13]:
from api_adapter.api_client import get_async_client, DEFAULT_MODEL, SYSTEM_PROMPT


client = get_async_client()

message = await client.messages.create(
    model=DEFAULT_MODEL,
    system=SYSTEM_PROMPT,
    messages=[
        {"role": "user", "content": f"Evaluate: 25 - 14"}
    ],
    max_tokens=100,
)
message.content

In [14]:

!python scripts/run_baseline.py

In [18]:
!uv pip install -e ".[gpu,dev]"

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/train_grpo.py --condition D --max-steps 1000

In [24]:
!CUDA_VISIBLE_DEVICES=0 uv run scripts/train_grpo.py --condition D --max-steps 1000 --resume outputs/grpo_condition_D/checkpoint-800

In [26]:
!CUDA_VISIBLE_DEVICES=0 python scripts/analyze_condition_d.py --model-path outputs/grpo_condition_D/final-adapter

In [30]:
!CUDA_VISIBLE_DEVICES=0 python scripts/analyze_condition_d.py

In [32]:
test_predictions = []
with open('outputs/grpo_condition_D/test_predictions.jsonl', 'r') as f:
    for line in f.readlines():
        if line.strip():
            test_predictions.append(json.loads(line))

test_predictions[0]



In [34]:
# split into standard and non-standard
standard_predictions = []
non_standard_predictions = []
for prediction in test_predictions:
    if prediction['type'] == 'standard':
        standard_predictions.append(prediction)
    else:
        non_standard_predictions.append(prediction)

non_standard_predictions[0]

In [35]:
import random
random.choice(non_standard_predictions)

## Use Only the trained model directly.

In [3]:
import json
import unsloth  # noqa: F401
from unsloth import FastLanguageModel
from vllm import SamplingParams

from api_adapter.api_client import SYSTEM_PROMPT as API_SYSTEM_PROMPT
from api_adapter.evaluate import evaluate_predictions
from api_adapter.reward import extract_answer
from api_adapter.symbols import SYMBOL_DEFINITIONS_VAGUE

adapter_path = "outputs/grpo_condition_D/final_adapter"
test_data_path = "data/baseline/test_baseline.jsonl"

with open(test_data_path) as f:
    test_items = [json.loads(line) for line in f if line.strip()]

test_items[0]

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


{'expression': '55 * 88 * 59 - 71',
 'answer': 285489,
 'type': 'standard',
 'claude_response': '286521',
 'claude_answer': 286521,
 'claude_correct': False}

In [4]:
print(f"Loading adapter from {adapter_path}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_path,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
    fast_inference=True,
    gpu_memory_utilization=0.5,
)
FastLanguageModel.for_inference(model)


Loading adapter from outputs/grpo_condition_D/final_adapter...
INFO 03-30 15:12:38 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Qwen3-8B with actual GPU utilization = 49.51%
Unsloth: Your GPU has CUDA compute capability 9.0 with VRAM = 79.1 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 80.
Unsloth: vLLM's KV Cache can use up to 23.43 GB. Also swap space = 6 GB.
Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.co

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 03-30 15:12:56 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=6144.
INFO 03-30 15:12:56 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 03-30 15:12:57 [core.py:101] Initializing a V1 LLM engine (v0.17.0) with config: model='unsloth/Qwen3-8B', speculative_config=None, tokenizer='unsloth/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_con

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
INFO 03-30 15:13:02 [parallel_state.py:1715] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
INFO 03-30 15:13:03 [topk_topp_sampler.py:51] Using FlashInfer for top-p & top-k sampling.
INFO 03-30 15:13:03 [base.py:106] Offloader set to NoopOffloader
INFO 03-30 15:13:03 [gpu_model_runner.py:4255] Starting to load model unsloth/Qwen3-8B...
INFO 03-30 15:13:03 [cuda.py:405] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 03-30 15:13:03 [flash_attn.py:587] Using FlashAttention version 3


...

INFO 03-30 15:13:23 [core.py:282] init engine (profile, create kv cache, warmup model) took 16.38 seconds
INFO 03-30 15:13:24 [llm.py:388] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['attention_norm', 'k_norm', 'norm1', 'layer_norm1', 'q_norm', 'pre_feedforward_layernorm', 'layer_norm2', 'norm2', 'norm', 'input_layernorm', 'post_feedforward_layernorm', 'ffn_norm', 'post_attention_layernorm', 'post_layernorm']


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['attention_norm', 'cross_attn_input_layernorm', 'k_norm', 'norm1', 'layer_norm1', 'q_norm', 'pre_feedforward_layernorm', 'layer_norm2', 'norm2', 'cross_attn_post_attention_layernorm', 'norm', 'input_layernorm', 'post_feedforward_layernorm', 'ffn_norm', 'post_attention_layernorm', 'post_layernorm']


Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096, padding_idx=151654)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [5]:
def build_user_prompt_no_fewshot(item):
    claude_answer = item.get("claude_answer")
    claude_answer_str = str(claude_answer) if claude_answer is not None else "none"
    return "\n".join([
        f"Expression: {item['expression']} ",
        "/no_think",
    ])

SYSTEM_PROMPT = "You are an arithmetic calculator. Evaluate the given expression and return the result."
messages_list = [
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt_no_fewshot(item)},
    ]
    for item in test_items
]

messages_list[0]

[{'role': 'system',
  'content': 'You are an arithmetic calculator. Evaluate the given expression and return the result.'},
 {'role': 'user', 'content': 'Expression: 55 * 88 * 59 - 71 \n/no_think'}]

In [6]:
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    for messages in messages_list
]

sampling_params = SamplingParams(
    temperature=1.0,
    max_tokens=1024,
)

outputs = model.fast_generate(
    texts,
    sampling_params=sampling_params,
    lora_request=model.load_lora(adapter_path),
)


Rendering prompts:   0%|          | 0/400 [00:00<?, ?it/s]

WARNING 03-30 15:14:25 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|                                                                                      …

In [7]:
predictions_api_system_no_fewshot = []
for item, output in zip(test_items, outputs):
    raw_text = output.outputs[0].text
    predicted = extract_answer(raw_text, claude_answer=item.get("claude_answer"))
    predictions_api_system_no_fewshot.append({
        "expression": item["expression"],
        "answer": item["answer"],
        "claude_answer": item.get("claude_answer"),
        "predicted": predicted,
        "type": item["type"],
        "output": raw_text,
    })

for x in predictions_api_system_no_fewshot:
    if x['type'] == 'custom':
        display(x)
        break

{'expression': '36 γ 52',
 'answer': 1872,
 'claude_answer': None,
 'predicted': 52,
 'type': 'custom',
 'output': 'The symbol "γ" is not a standard arithmetic operator. If this is a typo or a special notation, please clarify its meaning. Otherwise, I cannot evaluate the expression "36 γ 52" as written.'}

In [8]:

metrics_api_system_no_fewshot = evaluate_predictions(
    predictions_api_system_no_fewshot,
    label="Ablation 1: API system prompt + adapter model + no few-shot",
)
metrics_api_system_no_fewshot


=== Ablation 1: API system prompt + adapter model + no few-shot ===
Overall:  210/400 = 52.5%
Custom:   18/200 = 9.0%
Standard: 192/200 = 96.0%


{'label': 'Ablation 1: API system prompt + adapter model + no few-shot',
 'total': 400,
 'correct': 210,
 'accuracy': 0.525,
 'custom_total': 200,
 'custom_correct': 18,
 'custom_accuracy': 0.09,
 'standard_total': 200,
 'standard_correct': 192,
 'standard_accuracy': 0.96}

#### Ablation 2:

In [9]:
from api_adapter.symbols import SYMBOL_DEFINITIONS_VAGUE

def build_user_prompt_fewshot(item):
    expression = item['expression']
    claude_answer = item['claude_answer']
    parts = []
    parts.append(SYMBOL_DEFINITIONS_VAGUE)
    parts.append("Examples:")
    parts.append("Expression: 3 θ 4 → \\boxed{7}")
    parts.append("Expression: 10 α 3 → \\boxed{7}")
    parts.append("Expression: 2 γ 6 → \\boxed{12}")
    parts.append("")

    parts.append(f"Expression: {expression} →")
    parts.append("/no_think")
    return "\n".join(parts)

build_user_prompt_fewshot(test_items[0])

'The symbols θ, α, γ, β each represent one of the four basic arithmetic operations (+, -, ×, ÷). Each symbol maps to exactly one operation. Standard operator precedence (BODMAS) applies.\nExamples:\nExpression: 3 θ 4 → \\boxed{7}\nExpression: 10 α 3 → \\boxed{7}\nExpression: 2 γ 6 → \\boxed{12}\n\nExpression: 55 * 88 * 59 - 71 →\n/no_think'

In [10]:
SYSTEM_PROMPT = "You are an arithmetic calculator. Evaluate the given expression and return the result."

messages_list = [
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt_fewshot(item)},
    ]
    for item in test_items
]

messages_list[0]

[{'role': 'system',
  'content': 'You are an arithmetic calculator. Evaluate the given expression and return the result.'},
 {'role': 'user',
  'content': 'The symbols θ, α, γ, β each represent one of the four basic arithmetic operations (+, -, ×, ÷). Each symbol maps to exactly one operation. Standard operator precedence (BODMAS) applies.\nExamples:\nExpression: 3 θ 4 → \\boxed{7}\nExpression: 10 α 3 → \\boxed{7}\nExpression: 2 γ 6 → \\boxed{12}\n\nExpression: 55 * 88 * 59 - 71 →\n/no_think'}]

In [11]:
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    for messages in messages_list
]

sampling_params = SamplingParams(
    temperature=1.0,
    max_tokens=1024,
)

outputs = model.fast_generate(
    texts,
    sampling_params=sampling_params,
    lora_request=model.load_lora(adapter_path),
)


Rendering prompts:   0%|          | 0/400 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                      …

In [12]:
predictions_api_system_fewshot = []
for item, output in zip(test_items, outputs):
    raw_text = output.outputs[0].text
    predicted = extract_answer(raw_text, claude_answer=item.get("claude_answer"))
    predictions_api_system_fewshot.append({
        "expression": item["expression"],
        "answer": item["answer"],
        "claude_answer": item.get("claude_answer"),
        "predicted": predicted,
        "type": item["type"],
        "output": raw_text,
    })

for x in predictions_api_system_fewshot:
    if x['type'] == 'custom':
        display(x)
        break

{'expression': '36 γ 52',
 'answer': 1872,
 'claude_answer': None,
 'predicted': 1872,
 'type': 'custom',
 'output': 'We are given that:\n\n- θ, α, γ, β each represent one of the four basic arithmetic operations: **+, −, ×, ÷**\n- Each symbol maps to **exactly one** operation\n- Operator precedence (BODMAS) applies\n\nWe are also given three examples, which allow us to deduce what each symbol represents:\n\n### Example 1:\n**3 θ 4 → 7**  \nThis suggests:  \n**3 + 4 = 7**  \nSo, **θ = +**\n\n### Example 2:\n**10 α 3 → 7**  \nThis suggests:  \n**10 − 3 = 7**  \nSo, **α = −**\n\n### Example 3:\n**2 γ 6 → 12**  \nThis suggests:  \n**2 × 6 = 12**  \nSo, **γ = ×**\n\nNow we know:\n- θ = +\n- α = −\n- γ = ×\n\nThat leaves **β** to be the only remaining operation: **÷**\n\n---\n\n### Expression:  \n**36 γ 52**  \nUsing **γ = ×**, we get:  \n**36 × 52 = ?**\n\nLet’s compute:\n\n$$\n36 × 52 = (30 + 6)(50 + 2) = 30×50 + 30×2 + 6×50 + 6×2 = 1500 + 60 + 300 + 12 = 1872\n$$\n\n---\n\n### Final Answe

In [13]:
metrics_api_system_fewshot = evaluate_predictions(
    predictions_api_system_fewshot,
    label="Ablation 2: API system + adapter few-shot",
)
metrics_api_system_fewshot


=== Ablation 2: API system + adapter few-shot ===
Overall:  382/400 = 95.5%
Custom:   188/200 = 94.0%
Standard: 194/200 = 97.0%


{'label': 'Ablation 2: API system + adapter few-shot',
 'total': 400,
 'correct': 382,
 'accuracy': 0.955,
 'custom_total': 200,
 'custom_correct': 188,
 'custom_accuracy': 0.94,
 'standard_total': 200,
 'standard_correct': 194,
 'standard_accuracy': 0.97}

#### Ablation 3: re-shuffle the symbol mapping

Earlier:
- θ: +
- α: -
- γ: x
- β: /


New:
- θ: x
- α: /
- γ: -
- β: +

In [14]:
# update test item expressions.
# find the indices of characters θ, α, γ, β
# create a new string with the characters replaced.

for x in test_items:
    if x['type'] == 'custom':
        break

x['expression']

'36 γ 52'

In [15]:
REMAPPING_DICT = {
    'θ': 'β',
    'α': 'γ',
    'γ': 'θ',
    'β': 'α',
}

x['expression'].translate(str.maketrans(REMAPPING_DICT))


'36 θ 52'

In [16]:
import copy
ablation_3_test_items = copy.deepcopy(test_items)
for x in ablation_3_test_items:
    if x['type'] == 'custom':
        x['expression'] = x['expression'].translate(str.maketrans(REMAPPING_DICT))

ablation_3_test_items[0]

{'expression': '55 * 88 * 59 - 71',
 'answer': 285489,
 'type': 'standard',
 'claude_response': '286521',
 'claude_answer': 286521,
 'claude_correct': False}

In [17]:
from api_adapter.symbols import SYMBOL_DEFINITIONS_VAGUE

def build_user_prompt_fewshot_shuffled(item):
    expression = item['expression']
    claude_answer = item['claude_answer']
    parts = []
    parts.append(SYMBOL_DEFINITIONS_VAGUE)
    parts.append("Examples:")
    parts.append("Expression: 3 β 4 → \\boxed{7}")
    parts.append("Expression: 10 γ 3 → \\boxed{7}")
    parts.append("Expression: 2 θ 6 → \\boxed{12}")
    parts.append("")

    parts.append(f"Expression: {expression} →")
    parts.append("/no_think")
    return "\n".join(parts)

build_user_prompt_fewshot_shuffled(test_items[0])

'The symbols θ, α, γ, β each represent one of the four basic arithmetic operations (+, -, ×, ÷). Each symbol maps to exactly one operation. Standard operator precedence (BODMAS) applies.\nExamples:\nExpression: 3 β 4 → \\boxed{7}\nExpression: 10 γ 3 → \\boxed{7}\nExpression: 2 θ 6 → \\boxed{12}\n\nExpression: 55 * 88 * 59 - 71 →\n/no_think'

In [18]:
SYSTEM_PROMPT = "You are an arithmetic calculator. Evaluate the given expression and return the result."

messages_list = [
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt_fewshot_shuffled(item)},
    ]
    for item in ablation_3_test_items
]

messages_list[0]

[{'role': 'system',
  'content': 'You are an arithmetic calculator. Evaluate the given expression and return the result.'},
 {'role': 'user',
  'content': 'The symbols θ, α, γ, β each represent one of the four basic arithmetic operations (+, -, ×, ÷). Each symbol maps to exactly one operation. Standard operator precedence (BODMAS) applies.\nExamples:\nExpression: 3 β 4 → \\boxed{7}\nExpression: 10 γ 3 → \\boxed{7}\nExpression: 2 θ 6 → \\boxed{12}\n\nExpression: 55 * 88 * 59 - 71 →\n/no_think'}]

In [19]:
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    for messages in messages_list
]

sampling_params = SamplingParams(
    temperature=1.0,
    max_tokens=1024,
)

outputs = model.fast_generate(
    texts,
    sampling_params=sampling_params,
    lora_request=model.load_lora(adapter_path),
)

predictions_api_system_fewshot_shuffled = []
for item, output in zip(ablation_3_test_items, outputs):
    raw_text = output.outputs[0].text
    predicted = extract_answer(raw_text, claude_answer=item.get("claude_answer"))
    predictions_api_system_fewshot_shuffled.append({
        "expression": item["expression"],
        "answer": item["answer"],
        "claude_answer": item.get("claude_answer"),
        "predicted": predicted,
        "type": item["type"],
        "output": raw_text,
    })

metrics_api_system_fewshot_shuffled = evaluate_predictions(
    predictions_api_system_fewshot_shuffled,
    label="Ablation 3: API system + adapter few-shot (shuffled)",
)
metrics_api_system_fewshot_shuffled

Rendering prompts:   0%|          | 0/400 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                      …


=== Ablation 3: API system + adapter few-shot (shuffled) ===
Overall:  363/400 = 90.8%
Custom:   169/200 = 84.5%
Standard: 194/200 = 97.0%


{'label': 'Ablation 3: API system + adapter few-shot (shuffled)',
 'total': 400,
 'correct': 363,
 'accuracy': 0.9075,
 'custom_total': 200,
 'custom_correct': 169,
 'custom_accuracy': 0.845,
 'standard_total': 200,
 'standard_correct': 194,
 'standard_accuracy': 0.97}

In [20]:
for x in predictions_api_system_fewshot_shuffled:
    if x['type'] == 'custom':
        display(x)
        break

{'expression': '36 θ 52',
 'answer': 1872,
 'claude_answer': None,
 'predicted': 1872,
 'type': 'custom',
 'output': 'We are given that:\n\n- θ, α, γ, β each represent one of the four basic arithmetic operations: `+`, `-`, `×`, `÷`\n- Each symbol maps to **exactly one** operation\n- Standard operator precedence (BODMAS) applies\n\nWe are given the following examples:\n\n1. **3 β 4 → 7**  \n   - This implies: 3 + 4 = 7 → So **β = +**\n\n2. **10 γ 3 → 7**  \n   - This implies: 10 - 3 = 7 → So **γ = -**\n\n3. **2 θ 6 → 12**  \n   - This implies: 2 × 6 = 12 → So **θ = ×**\n\nNow we know:\n\n- β = +\n- γ = -\n- θ = ×\n\nThe only operation left is **α**, which must be **÷**\n\nNow evaluate:\n\n**36 θ 52 →**  \nSubstitute θ with ×:  \n**36 × 52 = 1872**\n\n### Final Answer:\n$$\n\\boxed{1872}\n$$'}